## Training Stacking Ensemble Learning

<table>
        <thead>
            <tr>
                <th rowspan="2">Modelo</th>
                <th rowspan="2">Acurácia</th>
                <th colspan="2">Precisão</th>
                <th colspan="2">Recall</th>
                <th colspan="2">F1-Score</th>
            </tr>
            <tr>
                <th>Classe 0</th>
                <th>Classe 1</th>
                <th>Classe 0</th>
                <th>Classe 1</th>
                <th>Classe 0</th>
                <th>Classe 1</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>NB - All</td>
                <td>64%</td>
                <td>0.46</td>
                <td>0.83</td>
                <td>0.73</td>
                <td>0.60</td>
                <td>0.56</td>
                <td>0.70</td>
            </tr>
            <tr>
                <td>NB - Selected</td>
                <td>69%</td>
                <td>0.50</td>
                <td>0.82</td>
                <td>0.67</td>
                <td>0.70</td>
                <td>0.58</td>
                <td>0.75</td>
            </tr>
</table>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##### Recovering the data

In [3]:
X_train = pd.read_csv("../../datasets/opckd_datasets/X_train.csv")
X_test  = pd.read_csv("../../datasets/opckd_datasets/X_test.csv")
y_train = pd.read_csv("../../datasets/opckd_datasets/y_train.csv").values.ravel()
y_test  = pd.read_csv("../../datasets/opckd_datasets/y_test.csv").values.ravel()

In [4]:
print("Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Shapes:
X_train: (7242, 27)
X_test: (1323, 27)
y_train: (7242,)
y_test: (1323,)


#### Training the model with all variables

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("stack", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
            ("et", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
            ("gb", GradientBoostingClassifier(random_state=42)),
            ("knn", KNeighborsClassifier())
        ],
        final_estimator=LogisticRegression(max_iter=3000),
        cv=5,
        n_jobs=-1,
        passthrough=False
    ))
])

param_grid = {
    "stack__rf__n_estimators": [100, 200],
    "stack__et__n_estimators": [100, 200],
    "stack__gb__n_estimators": [50, 100],
    "stack__final_estimator__C": [0.1, 1.0, 10.0]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Melhores hiperparâmetros encontrados:")
print(grid.best_params_)
print("\nMelhor score de validação:", grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("\n==================== MÉTRICAS MODELO FINAL ====================")
print(f"Acurácia:   {acc:.4f}")
print(f"Precisão:   {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1-score:   {f1:.4f}")
print("\n==================== CLASSIFICATION REPORT ====================")
print(classification_report(y_test, y_pred, zero_division=0))

Melhores hiperparâmetros encontrados:
{'stack__et__n_estimators': 200, 'stack__final_estimator__C': 10.0, 'stack__gb__n_estimators': 50, 'stack__rf__n_estimators': 100}

Melhor score de validação: 0.8605576717034801

==================== MÉTRICAS MODELO FINAL ====================
Acurácia:   0.7649
Precisão:   0.7552
Recall:     0.7649
F1-score:   0.7547

==================== CLASSIFICATION REPORT ====================
              precision    recall  f1-score   support

           0       0.67      0.50      0.58       418
           1       0.79      0.89      0.84       905

    accuracy                           0.76      1323
   macro avg       0.73      0.69      0.71      1323
weighted avg       0.76      0.76      0.75      1323



In [7]:
pipe = Pipeline([
    ("select", SelectKBest(score_func=f_classif)),
    ("stack", StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
            ("et", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
            ("gb", GradientBoostingClassifier(random_state=42)),
            ("knn", KNeighborsClassifier())
        ],
        final_estimator=LogisticRegression(max_iter=3000),
        cv=5,
        n_jobs=-1,
        passthrough=False
    ))
])

param_grid = {
    "select__k": [5, 10, 15, "all"],
    "stack__rf__n_estimators": [100, 200],
    "stack__et__n_estimators": [100, 200],
    "stack__gb__n_estimators": [50, 100],
    "stack__final_estimator__C": [0.1, 1.0, 10.0]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Melhores hiperparâmetros encontrados:")
print(grid.best_params_)
print("\nMelhor score de validação:", grid.best_score_)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

selector = best_model.named_steps["select"]
selected_features = X_train.columns[selector.get_support()]
print("\nFeatures selecionadas:")
print(list(selected_features))

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("\n==================== MÉTRICAS MODELO FINAL ====================")
print(f"Acurácia:   {acc:.4f}")
print(f"Precisão:   {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1-score:   {f1:.4f}")
print("\n==================== CLASSIFICATION REPORT ====================")
print(classification_report(y_test, y_pred, zero_division=0))

c:\Users\luizf\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [22] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\luizf\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Melhores hiperparâmetros encontrados:
{'select__k': 'all', 'stack__et__n_estimators': 200, 'stack__final_estimator__C': 10.0, 'stack__gb__n_estimators': 50, 'stack__rf__n_estimators': 100}

Melhor score de validação: 0.8605576717034801

Features selecionadas:
['Cre', 'eGFR', 'Age', 'BMI', 'Albumin', 'Calcium', 'Phosphate', 'ALP', 'HbA1c', 'Glucose', 'WBC', 'HGB', 'Gender', 'Steroid usage', 'OA', 'RA', 'DM', 'HTN', 'CVA', 'Cataract', 'Fragility fracture hx', 'T1DM', 'Osteogenesis Imperfecta', 'Hyperthyroidism', 'Hypogonadism', 'Malnutrition', 'Chronic liver disease']

==================== MÉTRICAS MODELO FINAL ====================
Acurácia:   0.7649
Precisão:   0.7552
Recall:     0.7649
F1-score:   0.7547

==================== CLASSIFICATION REPORT ====================
              precision    recall  f1-score   support

           0       0.67      0.50      0.58       418
           1       0.79      0.89      0.84       905

    accuracy                           0.76      1323
   